In [1]:
from transformers import (
    AutoTokenizer , AutoModelForCausalLM,
    TrainingArguments , Trainer
)
from peft import LoraConfig , get_peft_model , TaskType
from datasets import load_dataset

In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [5]:
tokenizer.eos_token

'</s>'

In [6]:
import zipfile
import os 
zip_path = "tinyllama-non-instruction.zip"

with zipfile.ZipFile(zip_path , 'r') as zip_ref:
    zip_ref.extractall()

In [7]:
model_path = "checkpoint-5"

non_instruction_model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    device_map = "auto"
)

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [8]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
inputs

{'input_ids': tensor([[    1,   315,  1915,   936,  3367,  1338, 28585,   393, 29299,  2180,
           272, 29894,   579, 21203,   411,   382,  4975,   326, 18673]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
       device='cuda:0')}

In [9]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens = 100,
    temperature = 0.8,
    top_p = 0.9,
    do_sample = True,
    repetition_penalty = 1.1
)

In [10]:
outputs

tensor([[    1,   315,  1915,   936,  3367,  1338, 28585,   393, 29299,  2180,
           272, 29894,   579, 21203,   411,   382,  4975,   326, 18673, 12212,
           278,   365, 19558, 29899, 29907,   313,   677,  9027,   619,  7323,
          4859,   262,   521,   324,  4156,   324, 29897,   491, 29871, 29906,
         29900, 29995,   297, 17800,   411,   263,  3517,  4955,   310,  5192,
         17135, 29889,    13,  1576,   383,  7698, 23454,  2180,   307,   794,
         22318,   284,  1706,   764,   363, 22069,  1058,   526,   599, 15064,
           293,   304,  8281,   284,   805,   764, 29879,   322,   363,  6029,
           470,   284, 13589,   800,  1316,   408,   379,  4519,   297,  4077,
           362,  3234, 29892,   379,  4519, 29899, 29896, 29941, 29946, 29874,
           297,  4077,   362,   805,   764, 29892,   470,   317,  1743, 29920,
         29915, 29879,  1019,   794,   309,   379,  4519,  8281,   284]],
       device='cuda:0')

In [11]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0] , skip_special_tokens = True))


Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe reduced the LDL-C (low density lipoprotein cholesterol) by 20% in subjects with a previous history of heart disease.
The FDA approved Atrovent Nasal Spray for patients who are allergic to nasal sprays and for whom oral medications such as HFA inhalation product, HFA-134a inhalation spray, or Sandoz's Proventil HFA nasal


In [12]:
dataset = load_dataset(
    "Amod/mental_health_counseling_conversations", 
    split = "train"
)
dataset

README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [13]:
def format_row(example):
    question = example["Context"]
    answer = example["Response"]
    example["Text"] = f"[Context] {question} [/Response] {answer}"
    return example

In [15]:
formated_dataset = dataset.map(format_row)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [16]:
formated_dataset

Dataset({
    features: ['Context', 'Response', 'Text'],
    num_rows: 3512
})

In [18]:
print(formated_dataset['Text'][0])

[Context] I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.
   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.
   How can I change my feeling of being worthless to everyone? [/Response] If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media.  Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.

In [22]:
def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

In [23]:
dataset = load_dataset(
    "csv" , data_files = "pharma_instruction_data.csv" ,split = 'train'
)
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})

In [24]:
dataset = dataset.map(format_example)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [25]:
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 5
})

In [26]:
dataset['text'][0]

'### Instruction:\nExplain the mechanism of action of Metformin.\n### Input:\nNone\n### Response:\nMetformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.'

In [27]:
def tokenize_fn(example):
    tokens = tokenizer(
        example["text"], 
        truncation = True, 
        padding="max_length", 
        max_length = 512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [28]:
tokenized = dataset.map(tokenize_fn, batched = True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [29]:
tokenized

Dataset({
    features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 5
})

In [30]:
lora_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r = 8,
    lora_alpha = 16,
    lora_dropout = 0.05,
    target_modules = ["q_proj", "v_proj"],
    bias = "none"
)

In [32]:
type(non_instruction_model)

transformers.models.llama.modeling_llama.LlamaForCausalLM

In [34]:
instruction_model = get_peft_model(non_instruction_model ,peft_config = lora_config)

c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\peft\mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
c:\Users\tipto\anaconda3\envs\nlp\lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [35]:
args = TrainingArguments(
    output_dir = "./tinyllama-instruction",
    num_train_epochs = 2,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 20,
    save_total_limit = 1,
    report_to = "none"
)

In [36]:
trainer = Trainer(
    model = instruction_model,
    args = args,
    train_dataset = tokenized
)

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [38]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=2, training_loss=9.961847305297852, metrics={'train_runtime': 2.7085, 'train_samples_per_second': 3.692, 'train_steps_per_second': 0.738, 'total_flos': 31814823444480.0, 'train_loss': 9.961847305297852, 'epoch': 2.0})